# 05 — Bagging splits (N=7, 200-image disjoint hold-outs)

Escalation per `WBF_context.md` § Escalation: train 7 RT-DETR-X bags. Each bag holds out a unique
200-image slice (1,400 disjoint hold-outs total, leaving each bag's training set at 2,954 images —
matching Exp D's full-data budget far better than the k=5 fold's 2,523).

Why N=7 (vs the WBF_guidelines § N=5 default): NMS gains plateau around N=6–8. WBF underperformed on
this data so we're optimizing for NMS, where the marginal model past 7 rarely flips a vote. N=7 also
maximizes total hold-out coverage at 1,400 images = 44% of the train set, useful for ensemble diagnostics.

Reuses [01_folds.ipynb](01_folds.ipynb)'s `images/train/` junction and `labels/train/` YOLO label tree —
no rebuild needed. Outputs:
- `bags/bag_{0..6}_train.txt`, `bag_{0..6}_val.txt` — absolute image paths
- `bags/bag_{0..6}.yaml` — Ultralytics data configs
- `bags/bag_assignments.json` — `{bag_i: [held-out image_ids]}` for reproducibility / debugging

## Reproducibility

In [1]:
import random, numpy as np, torch
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

## Paths and load GT

Asserts the prerequisites from `01_folds.ipynb` (label tree, image junction) exist.

In [2]:
import json
from pathlib import Path
from collections import defaultdict

PROJECT_ROOT = Path(r"C:/Users/Victor/Desktop/Projects/ESA_SAR_WBF")
SRC_ROOT     = Path(r"C:/Users/Victor/Desktop/Projects/ESA_SAR/ClearSAR")
ANN_FILE     = SRC_ROOT / "data/annotations/instances_train.json"

IMAGES_MIRROR = PROJECT_ROOT / "images/train"
LABELS_DIR    = PROJECT_ROOT / "labels/train"
BAGS_DIR      = PROJECT_ROOT / "bags"
BAGS_DIR.mkdir(parents=True, exist_ok=True)

assert IMAGES_MIRROR.is_dir(), f"missing {IMAGES_MIRROR} — run 01_folds.ipynb first"
assert LABELS_DIR.is_dir(), f"missing {LABELS_DIR} — run 01_folds.ipynb first"

with open(ANN_FILE) as f:
    coco = json.load(f)
images = coco["images"]
ann_by_img = defaultdict(list)
for a in coco["annotations"]:
    ann_by_img[a["image_id"]].append(a)

print(f"images: {len(images)}, with annotations: {sum(1 for im in images if ann_by_img[im['id']])}")
assert len(images) == 3154

images: 3154, with annotations: 3120


## Sample 1,400 hold-outs and partition into 7 disjoint groups

Strategy: stratified 1,400-image hold-out pool (preserves 34/3,154 negative ratio → ~15 negatives in pool),
then `StratifiedKFold(7)` *on the pool* to split into 7 disjoint 200-image groups while spreading negatives
across bags. The 1,754 images never selected as hold-outs are in **every** bag's training set — that's by
design, this is bagging, not k-fold.

In [3]:
from sklearn.model_selection import train_test_split, StratifiedKFold

img_ids = [im["id"] for im in images]
stems   = [Path(im["file_name"]).stem for im in images]
y_strat = [1 if ann_by_img[iid] else 0 for iid in img_ids]
paths   = [str((IMAGES_MIRROR / f"{s}.png").as_posix()) for s in stems]

all_idx = np.arange(len(img_ids))
y_arr   = np.array(y_strat)

# Step 1: pick 1400 stratified hold-out indices.
holdout_idx, _ = train_test_split(all_idx, train_size=1400, stratify=y_arr, random_state=SEED)
holdout_y = y_arr[holdout_idx]
print(f"hold-out pool: {len(holdout_idx)}  negatives in pool: {(holdout_y == 0).sum()} / {(y_arr == 0).sum()}")

# Step 2: partition pool into 7 disjoint groups of 200, stratified on negatives.
skf = StratifiedKFold(n_splits=7, shuffle=True, random_state=SEED)
bag_holdouts = []
for _, group_local_idx in skf.split(holdout_idx, holdout_y):
    bag_holdouts.append(holdout_idx[group_local_idx])

for i, h in enumerate(bag_holdouts):
    n_neg = int((y_arr[h] == 0).sum())
    print(f"bag {i}: hold-out={len(h)}  negatives={n_neg}")

# Sanity: disjoint and union to the original pool.
all_held = np.concatenate(bag_holdouts)
assert len(set(all_held)) == 1400, "hold-outs not disjoint"
assert set(all_held) == set(holdout_idx), "hold-outs don't cover the pool"

hold-out pool: 1400  negatives in pool: 15 / 34
bag 0: hold-out=200  negatives=2
bag 1: hold-out=200  negatives=2
bag 2: hold-out=200  negatives=2
bag 3: hold-out=200  negatives=2
bag 4: hold-out=200  negatives=2
bag 5: hold-out=200  negatives=2
bag 6: hold-out=200  negatives=3


## Write bag files (train/val .txt + .yaml + assignments.json)

In [4]:
assignments = {}
for i, holdout in enumerate(bag_holdouts):
    holdout_set = set(holdout.tolist())
    train_paths = [paths[k] for k in all_idx if k not in holdout_set]
    val_paths   = [paths[k] for k in holdout]

    (BAGS_DIR / f"bag_{i}_train.txt").write_text("\n".join(train_paths))
    (BAGS_DIR / f"bag_{i}_val.txt").write_text("\n".join(val_paths))

    yaml_text = (
        f"path: {PROJECT_ROOT.as_posix()}\n"
        f"train: bags/bag_{i}_train.txt\n"
        f"val: bags/bag_{i}_val.txt\n"
        f"names:\n"
        f"  0: RFI\n"
    )
    (BAGS_DIR / f"bag_{i}.yaml").write_text(yaml_text)

    assignments[i] = sorted(int(img_ids[k]) for k in holdout)

(BAGS_DIR / "bag_assignments.json").write_text(json.dumps(assignments, indent=2))
print(f"wrote bags/bag_{{0..6}}_train.txt, _val.txt, .yaml + bag_assignments.json")

wrote bags/bag_{0..6}_train.txt, _val.txt, .yaml + bag_assignments.json


## Sanity checks

- Each bag's train + val partitions are disjoint and union to all 3,154 images.
- Each bag's train set is exactly 2,954 images.
- Each bag's val set is exactly 200 images.
- All val images have a YOLO label file on disk (reused from `01_folds.ipynb`).
- Cross-bag: the 7 bag val sets are disjoint and their union is exactly 1,400 images.

In [5]:
all_val_paths = set()
for i in range(7):
    tr = set((BAGS_DIR / f"bag_{i}_train.txt").read_text().splitlines())
    va = set((BAGS_DIR / f"bag_{i}_val.txt").read_text().splitlines())
    assert len(tr) == 2954, (i, len(tr))
    assert len(va) == 200, (i, len(va))
    assert tr.isdisjoint(va), f"bag {i}: train/val overlap"
    assert tr | va == set(paths), f"bag {i}: train ∪ val != all images"
    assert all_val_paths.isdisjoint(va), f"bag {i}: val overlaps an earlier bag's val"
    all_val_paths |= va

assert len(all_val_paths) == 1400

# Every val image has a label file on disk (reused tree)
missing = [p for p in all_val_paths if not (LABELS_DIR / f"{Path(p).stem}.txt").exists()]
assert not missing, f"missing {len(missing)} label files"

print("✓ each bag: train=2954, val=200, disjoint, covers all 3154")
print("✓ 7 val sets disjoint, union=1400")
print("✓ all 1400 val images have YOLO label files on disk")

✓ each bag: train=2954, val=200, disjoint, covers all 3154
✓ 7 val sets disjoint, union=1400
✓ all 1400 val images have YOLO label files on disk


## Next steps

1. In [02_train.ipynb](02_train.ipynb), set `BAG = 0` (and `FOLD` is ignored) → run. Repeat for `BAG = 1..6`,
   restarting the kernel between runs. Run dirs land at `runs/kfold/bag_{0..6}_rtdetr_x/`.
2. After all 7 bags train, the predict + ensemble notebooks ([03_predict.ipynb](03_predict.ipynb),
   [04_ensemble.ipynb](04_ensemble.ipynb)) need a small adjustment to point at `bag_*` runs and `bags/bag_{N}_val.txt`
   instead of `fold_*`. Ping me when bag training is done and I'll wire those up.
3. Per `WBF_context.md` § Escalation, fusion uses the same `iou_thr=0.25, conf_type='avg'` defaults — no
   tuning slice carved off here, the user accepted the trade-off.